# Generators


## Writing an Iterator

An iterator keeps the state of the traversal in its attributes: `__iter__` returns the iterator itself, and `__next__` gives the next item, or raises `StopIteration` when there are none left.

1. Write the class `Fibonacci(limit)`, an iterator over the Fibonacci numbers strictly below `limit`: `list(Fibonacci(20))` gives `[0, 1, 1, 2, 3, 5, 8, 13]`.
2. Create `sequence = Fibonacci(20)` and print `list(sequence)` twice. Why is the second list empty?
3. Write the class `Log(lines)`, whose `__iter__` method is a generator that yields the non-empty lines. With `log = Log(["a", "", "b"])`, print `list(log)` twice: both lists must be `["a", "b"]`. Why does the second traversal work this time?

In [ ]:
# Your code here
from collections.abc import Iterator


class Fibonacci:
  def __init__(self, limit: int) -> None: ...

  def __iter__(self) -> "Fibonacci": ...

  def __next__(self) -> int: ...


class Log:
  def __init__(self, lines: list[str]) -> None: ...

  def __iter__(self) -> Iterator[str]: ...

### Additional hints

`__init__` stores the limit and the state of the traversal in attributes: `self.current = 0`, `self.following = 1`. `__iter__` returns `self`. `__next__` raises `StopIteration` when `self.current` reaches the limit; otherwise, it keeps the value to return, moves both attributes forward (the new `following` is the sum of the two old ones), and returns that value. In `Log`, `__iter__` contains a `yield`: each traversal calls it, and gets a fresh generator.

```python
class Fibonacci:
  def __init__(self, limit: int) -> None:
    self.limit = limit
    self.current = 0
    self.following = 1

  def __iter__(self) -> "Fibonacci":
    return self

  def __next__(self) -> int:
    if self.current >= self.limit:
      raise ...
    value = self.current
    self.current, self.following = ...
    return value


class Log:
  def __init__(self, lines: list[str]) -> None:
    self.lines = lines

  def __iter__(self) -> Iterator[str]:
    for line in self.lines:
      if line:
        yield ...
```

### Solution

In [ ]:
from collections.abc import Iterator


class Fibonacci:
  def __init__(self, limit: int) -> None:
    self.limit = limit
    self.current = 0
    self.following = 1

  def __iter__(self) -> "Fibonacci":
    return self

  def __next__(self) -> int:
    if self.current >= self.limit:
      raise StopIteration
    value = self.current
    self.current, self.following = self.following, self.current + self.following
    return value


class Log:
  def __init__(self, lines: list[str]) -> None:
    self.lines = lines

  def __iter__(self) -> Iterator[str]:
    for line in self.lines:
      if line:
        yield line


sequence = Fibonacci(20)
print(list(sequence))  # [0, 1, 1, 2, 3, 5, 8, 13]
print(list(sequence))  # []

log = Log(["a", "", "b"])
print(list(log))  # ['a', 'b']
print(list(log))  # ['a', 'b']

`Fibonacci` is an iterator: `__iter__` returns `self`, and the second `list` starts from an exhausted state. `Log` is an iterable: each call of `__iter__` creates a new generator, with its own state, and the object only keeps the data.

## Infinite Date Generator

Write the function `gen_dates` that, starting from a start date included, generates all successive dates separated by one week. For a start on 1900-01-01, the first three dates are 1900-01-01, 1900-01-08 and 1900-01-15.

**Useful documentation**

- [`datetime.timedelta`](https://docs.python.org/3/library/datetime.html#datetime.timedelta)

In [ ]:
# Your code here
import datetime
from collections.abc import Iterator


def gen_dates(date: datetime.date) -> Iterator[datetime.date]: ...

Print the first 10 dates with a `for` loop, which you stop with `break`.

Then write `take(iterable, n)`, a generator that yields the first `n` items of an iterable without reading one more: with `numbers = iter(range(10))`, after `list(take(numbers, 3))`, `next(numbers)` must give 3. Print the same 10 dates with `take`, then with `itertools.islice`.

In [ ]:
# Your code here
def take(iterable, n): ...

### Additional hints

`datetime.timedelta(days=7)` represents a week, and can be added to a date. In `gen_dates`, yield the current date before moving it forward: in the opposite order, the start date is never yielded. For `take`, `zip` stops as soon as one of its arguments is exhausted, and it reads them in order: `zip(range(n), iterable)` sees `range` stop before touching the iterable, whereas `zip(iterable, range(n))` reads one item too many. Expected: 10 dates, from 1900-01-01 to 1900-03-05.

```python
def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  week = datetime.timedelta(days=7)
  while True:
    yield ...
    date += ...


def take(iterable, n):
  for _, item in zip(range(n), iterable):
    yield ...


for date in take(gen_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

for date in itertools.islice(..., 10):
  print(date)
```

### Solution

In [ ]:
import datetime
import itertools
from collections.abc import Iterable, Iterator


def gen_dates(date: datetime.date) -> Iterator[datetime.date]:
  week = datetime.timedelta(days=7)
  while True:
    yield date
    date += week


for i, d in enumerate(gen_dates(datetime.date(1900, 1, 1))):
  if i == 10:
    break
  print(d)


def take(iterable: Iterable, n: int) -> Iterator:
  for _, item in zip(range(n), iterable):
    yield item


for date in take(gen_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

for date in itertools.islice(gen_dates(datetime.date(1900, 1, 1)), 10):
  print(date)

numbers = iter(range(10))
print(list(take(numbers, 3)), next(numbers))  # [0, 1, 2] 3

## Lazy Pipeline over a Large File

Generators chain transformations without ever storing the intermediate data. The next cell creates a log of 500,000 requests (`number;user;duration_ms`) and provides `peak_memory`, which measures the peak memory of a call.

1. Write three generators: `read_lines(path)`, which yields the lines of the file, `parse(lines)`, which yields `(user, duration)` tuples, and `slow(records, threshold)`, which only keeps requests longer than `threshold`.
2. Write `total_slow_generators(path)`, which chains your generators and returns, with `sum`, the total duration of the requests longer than 900 ms.
3. Write `total_slow_lists(path)`, which computes the same thing but stores each step in a list (`readlines`, `list(...)`).
4. Compare the results and the peak memory of both versions with `peak_memory`. The totals must be equal, and the list-based version should use about a hundred megabytes more.

In [ ]:
import random
import tracemalloc

with open("requests.log", "w", encoding="utf8") as fh:
  for i in range(500_000):
    fh.write(f"{i};user{random.randrange(1_000)};{random.randrange(1_000)}\n")


def peak_memory(function):
  """Run function() and return its result and its peak memory in MB."""
  tracemalloc.start()
  result = function()
  _, peak = tracemalloc.get_traced_memory()
  tracemalloc.stop()
  return result, peak / 1e6

In [ ]:
# Your code here
from collections.abc import Iterable, Iterator


def read_lines(path: str) -> Iterator[str]: ...


def parse(lines: Iterable[str]) -> Iterator[tuple[str, int]]: ...


def slow(
  records: Iterable[tuple[str, int]], threshold: int
) -> Iterator[tuple[str, int]]: ...


def total_slow_generators(path: str) -> int: ...


def total_slow_lists(path: str) -> int: ...

### Additional hints

Each generator goes through what it receives with `for` and yields with `yield`; `read_lines` opens the file in a `with` and delegates to `fh` with `yield from`. `line.rstrip("\n").split(";")` splits a line into its three fields. `peak_memory` takes a function with no argument, hence the `lambda`, and returns that function's result and its peak memory. Expected: `True`, less than 0.1 MB for the generators, about a hundred MB for the lists.

```python
def read_lines(path: str) -> Iterator[str]:
  with open(path, encoding="utf8") as fh:
    yield from ...


def parse(lines: Iterable[str]) -> Iterator[tuple[str, int]]:
  for line in lines:
    _, user, duration = ...
    yield ...


def slow(
  records: Iterable[tuple[str, int]], threshold: int
) -> Iterator[tuple[str, int]]:
  for user, duration in records:
    if ...:
      yield user, duration


def total_slow_generators(path: str) -> int:
  return sum(duration for _, duration in ...)


def total_slow_lists(path: str) -> int:
  with open(path, encoding="utf8") as fh:
    lines = fh.readlines()
  records = list(...)
  return sum(duration for _, duration in list(...))


total_g, peak_g = peak_memory(lambda: total_slow_generators("requests.log"))
total_l, peak_l = ...
print(total_g == total_l, peak_g, peak_l)
```

### Solution

In [ ]:
from collections.abc import Iterable, Iterator


def read_lines(path: str) -> Iterator[str]:
  with open(path, encoding="utf8") as fh:
    yield from fh


def parse(lines: Iterable[str]) -> Iterator[tuple[str, int]]:
  for line in lines:
    _, user, duration = line.rstrip("\n").split(";")
    yield user, int(duration)


def slow(
  records: Iterable[tuple[str, int]], threshold: int
) -> Iterator[tuple[str, int]]:
  for user, duration in records:
    if duration > threshold:
      yield user, duration


def total_slow_generators(path: str) -> int:
  return sum(duration for _, duration in slow(parse(read_lines(path)), 900))


def total_slow_lists(path: str) -> int:
  with open(path, encoding="utf8") as fh:
    lines = fh.readlines()
  records = list(parse(lines))
  return sum(duration for _, duration in list(slow(records, 900)))


total_g, peak_g = peak_memory(lambda: total_slow_generators("requests.log"))
total_l, peak_l = peak_memory(lambda: total_slow_lists("requests.log"))
print(total_g == total_l, f"generators: {peak_g:.1f} MB, lists: {peak_l:.1f} MB")

The generators only keep one line in memory at a time, whatever the size of the file: the list-based version stores the whole file, then every tuple.

## Releasing a Resource with `close` (hard)

A generator can hold a resource (file, connection…) while it yields its values.

Write a generator `numbered_lines(path)` that prints `opening`, opens the file and yields its numbered lines `(number, line)` with `yield from`, then prints `closing` when the file is closed again, including when the caller stops before the end.

Read the first three lines of `requests.log`, the file created in the previous exercise, with `next`, then call `close()` on the generator: `closing` must be printed right away. Why?

**Useful documentation**

- [`generator.close`](https://docs.python.org/3/builtins/stdtypes.html#generator.close)
- [`GeneratorExit`](https://docs.python.org/3/library/exceptions.html#GeneratorExit)

In [ ]:
# Your code here
from collections.abc import Iterator


def numbered_lines(path: str) -> Iterator[tuple[int, str]]: ...

### Additional hints

`try`/`finally` surrounds the `with`: the `finally` block runs as soon as the generator ends, whether it has read everything or has been interrupted. `enumerate(fh, start=1)` numbers the lines from 1, and `yield from` yields each tuple. Expected: `opening`, the tuples numbered 1, 2 and 3, then `closing`.

```python
def numbered_lines(path: str) -> Iterator[tuple[int, str]]:
  print("opening")
  try:
    with open(path, encoding="utf8") as fh:
      yield from ...
  finally:
    ...


lines = numbered_lines("requests.log")
for _ in range(3):
  print(...)
lines.close()
```

### Solution

In [ ]:
from collections.abc import Iterator


def numbered_lines(path: str) -> Iterator[tuple[int, str]]:
  print("opening")
  try:
    with open(path, encoding="utf8") as fh:
      yield from enumerate(fh, start=1)
  finally:
    print("closing")


lines = numbered_lines("requests.log")
for _ in range(3):
  print(next(lines))
lines.close()

`close()` raises `GeneratorExit` at the generator's pause point: the exception propagates, the `with` closes the file, then the `finally` block around it prints `closing`. Without `close()`, the file would only be closed when the generator is destroyed.

## Using the `send` method of a generator (very advanced)

Write the generator `interests(amount)`. It yields the current amount, receives an interest rate through `send` (0.05 for 5%), and yields the new amount after applying the rate.

Create `amounts = interests(100)` and start it with `next`, which must give 100. Then send it the rates 0.05, 0.08 and -0.02: `send` must give about 105.0, 113.4 and 111.132.

**Useful documentation**

- [`generator.send`](https://docs.python.org/3/builtins/stdtypes.html#generator.send)

In [ ]:
# Your code here
from collections.abc import Generator


def interests(amount: float) -> Generator[float, float, None]: ...

### Additional hints

`rate = yield amount` does two things: it yields `amount` and pauses, then evaluates, when resumed, to the value sent by `send`. `send` returns the value of the next `yield`. A generator that has just been created isn't stopped on any `yield`: the initial `next` brings it to the first one, otherwise `send` raises `TypeError`. A rate of 5% multiplies the amount by `1 + rate`.

```python
def interests(amount: float) -> Generator[float, float, None]:
  while True:
    rate = yield ...
    amount *= ...


amounts = interests(100)
print(next(amounts))
for rate in [0.05, 0.08, -0.02]:
  print(amounts.send(rate))
```

### Solution

In [ ]:
from collections.abc import Generator


def interests(amount: float) -> Generator[float, float, None]:
  while True:
    rate = yield amount
    amount *= 1 + rate


amounts = interests(100)
print(next(amounts))  # Starts the generator: 100
for rate in [0.05, 0.08, -0.02]:
  print(amounts.send(rate))  # About 105.0, 113.4, 111.132